ไม่ดีกว่าเเล้ว

In [2]:
import cv2
import numpy as np
from IPython.display import display, Image, clear_output
import time
from collections import deque, defaultdict
from insightface.app import FaceAnalysis
from pytubefix import YouTube
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cosine

# =============================
# CONFIG
# =============================
YOUTUBE_URL = "https://youtu.be/cmdMopdk6lo"
START_TIME = 75
END_TIME = 120

TARGET_WIDTH = 800
FACE_PANEL_WIDTH = 150
FACE_SIZE = 120
MAX_FACES_DISPLAY = 6

FONT = cv2.FONT_HERSHEY_SIMPLEX

# Balanced parameters for close faces
MAX_DISAPPEARED = 300  # รอ 10 วินาที
REID_MEMORY = 600  # จำได้ 20 วินาที
MIN_CONFIDENCE = 0.3
FEATURE_THRESHOLD = 0.40  # เพิ่มขึ้นเพื่อป้องกัน false match
REID_THRESHOLD = 0.45  # เข้มงวดขึ้น
MIN_FACE_SIZE = 25

# Spatial constraints (KEY for preventing ID swap)
MAX_MOVEMENT_PER_FRAME = 150  # pixels - ไม่ให้กระโดดไกลเกินนี้
MAX_SIZE_CHANGE_RATIO = 2.0  # ขนาดไม่เปลี่ยนมากเกินเท่านี้
SPATIAL_WEIGHT_CLOSE_RANGE = 100  # pixels - ถ้าใกล้กว่านี้ให้เน้น spatial มาก

# Gallery & temporal smoothing
GALLERY_SIZE = 40
TEMPORAL_WINDOW = 8
QUALITY_THRESHOLD = 0.4

# Scene change detection
SCENE_CHANGE_THRESHOLD = 0.5

# Occlusion handling
MIN_VISIBILITY_FRAMES = 3  # ต้องเห็นติดต่อกันอย่างน้อย 3 frames

# =============================
# INIT INSIGHTFACE
# =============================
print("Loading InsightFace ArcFace model...")
app = FaceAnalysis(
    name="buffalo_l",
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
)
app.prepare(ctx_id=0, det_size=(640, 640))

# =============================
# UTILITY FUNCTIONS
# =============================
def extract_face(frame, box, margin=25):
    """Extract face with margin"""
    h, w = frame.shape[:2]
    x1, y1, x2, y2 = map(int, box)
    x1 = max(0, x1 - margin)
    y1 = max(0, y1 - margin)
    x2 = min(w, x2 + margin)
    y2 = min(h, y2 + margin)
    
    crop = frame[y1:y2, x1:x2]
    return crop if crop.size else None

def cosine_similarity(a, b):
    """Cosine similarity (0-1, higher=more similar)"""
    return 1 - cosine(a, b)

def bbox_center(box):
    """Get center of bounding box"""
    return np.array([(box[0] + box[2]) / 2, (box[1] + box[3]) / 2])

def bbox_size(box):
    """Get area of bounding box"""
    return (box[2] - box[0]) * (box[3] - box[1])

def bbox_iou(box1, box2):
    """Calculate IoU between two boxes"""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = bbox_size(box1)
    area2 = bbox_size(box2)
    union = area1 + area2 - inter
    
    return inter / union if union > 0 else 0

def face_quality_score(face_data):
    """Calculate face quality"""
    box = face_data['box']
    size = bbox_size(box)
    score = face_data.get('score', 0.5)
    
    size_score = min(np.sqrt(size) / 100.0, 1.0)
    quality = 0.5 * score + 0.5 * size_score
    return quality

def detect_scene_change(prev_frame, curr_frame):
    """Detect if scene changed"""
    if prev_frame is None:
        return False
    
    prev_small = cv2.resize(prev_frame, (160, 90))
    curr_small = cv2.resize(curr_frame, (160, 90))
    
    diff = cv2.absdiff(prev_small, curr_small)
    diff_score = np.mean(diff) / 255.0
    
    return diff_score > SCENE_CHANGE_THRESHOLD

def check_spatial_consistency(det_box, track_box, det_center, track_center):
    """
    Check if detection is spatially consistent with track
    Returns: (is_consistent, movement_distance, size_ratio)
    """
    # Check movement distance
    movement = np.linalg.norm(det_center - track_center)
    
    # Check size change
    det_size = bbox_size(det_box)
    track_size = bbox_size(track_box)
    size_ratio = max(det_size, track_size) / (min(det_size, track_size) + 1e-6)
    
    # Spatial consistency criteria
    is_consistent = (
        movement < MAX_MOVEMENT_PER_FRAME and
        size_ratio < MAX_SIZE_CHANGE_RATIO
    )
    
    return is_consistent, movement, size_ratio

# =============================
# TEMPORAL FEATURE GALLERY
# =============================
class TemporalFeatureGallery:
    """Manages temporal smoothed features"""
    
    def __init__(self, gallery_size=GALLERY_SIZE, temporal_window=TEMPORAL_WINDOW):
        self.galleries = defaultdict(list)
        self.temporal_features = defaultdict(deque)
        self.gallery_size = gallery_size
        self.temporal_window = temporal_window
        
    def add(self, track_id, embedding, quality, frame_num):
        """Add new embedding with temporal smoothing"""
        self.temporal_features[track_id].append(embedding)
        if len(self.temporal_features[track_id]) > self.temporal_window:
            self.temporal_features[track_id].popleft()
        
        # Calculate smoothed embedding
        if len(self.temporal_features[track_id]) > 0:
            weights = np.linspace(0.5, 1.0, len(self.temporal_features[track_id]))
            weights = weights / weights.sum()
            
            smoothed = np.zeros_like(embedding)
            for w, emb in zip(weights, self.temporal_features[track_id]):
                smoothed += w * emb
            smoothed = smoothed / (np.linalg.norm(smoothed) + 1e-8)
        else:
            smoothed = embedding
        
        self.galleries[track_id].append((smoothed, quality, frame_num))
        
        if len(self.galleries[track_id]) > self.gallery_size:
            self.galleries[track_id].sort(key=lambda x: x[1], reverse=True)
            self.galleries[track_id] = self.galleries[track_id][:self.gallery_size]
    
    def get_representative(self, track_id):
        """Get best quality embedding"""
        if track_id not in self.galleries or len(self.galleries[track_id]) == 0:
            return None
        return self.galleries[track_id][0][0]
    
    def compare(self, track_id, embedding, top_k=5):
        """Compare with top-k best embeddings"""
        if track_id not in self.galleries or len(self.galleries[track_id]) == 0:
            return 0.0
        
        k = min(top_k, len(self.galleries[track_id]))
        top_embeddings = [emb for emb, _, _ in self.galleries[track_id][:k]]
        
        similarities = [cosine_similarity(embedding, emb) for emb in top_embeddings]
        similarities.sort(reverse=True)
        
        # Weighted average
        weights = np.linspace(0.5, 1.0, len(similarities))
        weights = weights / weights.sum()
        
        return np.sum(np.array(similarities) * weights)

# =============================
# SPATIAL-AWARE FACE TRACKER
# =============================
class SpatialAwareFaceTracker:
    """Face tracker with strong spatial consistency"""
    
    def __init__(self):
        self.next_id = 1
        self.active_tracks = {}
        self.deleted_tracks = {}
        self.gallery = TemporalFeatureGallery()
        self.colors = {}
        self.frame_count = 0
        self.prev_frame = None
        self.scene_changed = False
        
        # Track history for spatial consistency
        self.position_history = defaultdict(deque)  # track_id -> positions
        self.size_history = defaultdict(deque)  # track_id -> sizes
        
    def _get_color(self, track_id):
        """Get consistent color"""
        if track_id not in self.colors:
            np.random.seed(track_id * 1234)
            self.colors[track_id] = tuple(map(int, np.random.randint(80, 255, 3)))
        return self.colors[track_id]
    
    def _cleanup_deleted_tracks(self):
        """Remove very old deleted tracks"""
        to_remove = []
        for track_id, data in self.deleted_tracks.items():
            if self.frame_count - data['deleted_frame'] > REID_MEMORY:
                to_remove.append(track_id)
        
        for track_id in to_remove:
            del self.deleted_tracks[track_id]
            if track_id in self.position_history:
                del self.position_history[track_id]
            if track_id in self.size_history:
                del self.size_history[track_id]
    
    def _predict_position(self, track_id):
        """Predict next position based on velocity"""
        if track_id not in self.position_history:
            return None
        
        history = list(self.position_history[track_id])
        if len(history) < 2:
            return history[-1] if history else None
        
        # Simple linear prediction
        velocity = history[-1] - history[-2]
        predicted = history[-1] + velocity
        return predicted
    
    def update(self, detections, frame):
        """Update tracks with strong spatial consistency"""
        self.frame_count += 1
        
        # Detect scene change
        self.scene_changed = detect_scene_change(self.prev_frame, frame)
        self.prev_frame = frame.copy()
        
        # Reset spatial history on scene change
        if self.scene_changed:
            self.position_history.clear()
            self.size_history.clear()
        
        self._cleanup_deleted_tracks()
        
        # No detections
        if len(detections) == 0:
            for track_id in list(self.active_tracks.keys()):
                self.active_tracks[track_id]['disappeared'] += 1
                if self.active_tracks[track_id]['disappeared'] > MAX_DISAPPEARED:
                    self._move_to_deleted(track_id)
            return
        
        # No active tracks - create or resurrect
        if len(self.active_tracks) == 0:
            for det in detections:
                matched_id = self._try_global_match(det)
                if matched_id is not None:
                    self._resurrect_or_update(matched_id, det)
                else:
                    self._create_track(det)
            return
        
        # Build cost matrix with STRONG spatial weighting
        track_ids = list(self.active_tracks.keys())
        cost_matrix = np.full((len(detections), len(track_ids)), 10.0)  # High default cost
        
        for i, det in enumerate(detections):
            det_center = bbox_center(det['box'])
            
            for j, track_id in enumerate(track_ids):
                track = self.active_tracks[track_id]
                track_center = bbox_center(track['box'])
                
                # Check spatial consistency
                is_consistent, movement, size_ratio = check_spatial_consistency(
                    det['box'], track['box'], det_center, track_center
                )
                
                # If NOT spatially consistent, severely penalize
                if not is_consistent and track['disappeared'] == 0:
                    cost_matrix[i, j] = 5.0  # Very high cost to prevent match
                    continue
                
                # Calculate individual scores
                iou = bbox_iou(det['box'], track['box'])
                feature_sim = self.gallery.compare(track_id, det['emb'])
                
                # Spatial score with distance penalty
                distance = np.linalg.norm(det_center - track_center)
                
                # Use predicted position if available
                predicted_pos = self._predict_position(track_id)
                if predicted_pos is not None and track['disappeared'] == 0:
                    predicted_distance = np.linalg.norm(det_center - predicted_pos)
                    distance = min(distance, predicted_distance)
                
                # Strong spatial weighting for close faces
                if distance < SPATIAL_WEIGHT_CLOSE_RANGE:
                    spatial_score = max(0, 1.0 - distance / SPATIAL_WEIGHT_CLOSE_RANGE)
                else:
                    spatial_score = max(0, 1.0 - distance / 300.0)
                
                # Size consistency bonus
                size_bonus = 1.0 / size_ratio if size_ratio > 1.0 else 1.0
                
                # Adaptive weighting based on context
                if self.scene_changed:
                    # After scene change - rely more on features
                    weights = [0.15, 0.70, 0.10, 0.05]
                elif track['disappeared'] > 0:
                    # Lost track - balanced
                    weights = [0.20, 0.50, 0.20, 0.10]
                elif distance < SPATIAL_WEIGHT_CLOSE_RANGE:
                    # Close faces - STRONG spatial priority
                    weights = [0.20, 0.30, 0.45, 0.05]
                else:
                    # Normal tracking
                    weights = [0.25, 0.45, 0.25, 0.05]
                
                combined_score = (
                    weights[0] * iou +
                    weights[1] * feature_sim +
                    weights[2] * spatial_score +
                    weights[3] * size_bonus
                )
                
                cost_matrix[i, j] = 1.0 - combined_score
        
        # Hungarian algorithm
        det_indices, track_indices = linear_sum_assignment(cost_matrix)
        
        matched_detections = set()
        matched_tracks = set()
        
        # Process matches with strict threshold
        for det_idx, track_idx in zip(det_indices, track_indices):
            cost = cost_matrix[det_idx, track_idx]
            similarity = 1.0 - cost
            
            # Strict threshold to prevent false matches
            threshold = FEATURE_THRESHOLD
            if self.scene_changed:
                threshold -= 0.05
            
            if similarity > threshold and cost < 1.5:  # Additional sanity check
                track_id = track_ids[track_idx]
                self._update_track(track_id, detections[det_idx])
                matched_detections.add(det_idx)
                matched_tracks.add(track_id)
        
        # Unmatched detections - try global match carefully
        for i, det in enumerate(detections):
            if i not in matched_detections:
                matched_id = self._try_global_match_with_spatial(det, matched_tracks)
                
                if matched_id is not None:
                    self._resurrect_or_update(matched_id, det)
                    matched_detections.add(i)
                    if matched_id in self.active_tracks:
                        matched_tracks.add(matched_id)
                else:
                    self._create_track(det)
        
        # Unmatched tracks
        for track_id in track_ids:
            if track_id not in matched_tracks:
                self.active_tracks[track_id]['disappeared'] += 1
                if self.active_tracks[track_id]['disappeared'] > MAX_DISAPPEARED:
                    self._move_to_deleted(track_id)
    
    def _try_global_match_with_spatial(self, detection, exclude_ids):
        """Try global match but check spatial consistency"""
        det_center = bbox_center(detection['box'])
        best_id = None
        best_score = 0.0
        
        # Check active tracks (not yet matched)
        for track_id in self.active_tracks.keys():
            if track_id in exclude_ids:
                continue
            
            track = self.active_tracks[track_id]
            track_center = bbox_center(track['box'])
            
            # Check spatial consistency
            is_consistent, _, _ = check_spatial_consistency(
                detection['box'], track['box'], det_center, track_center
            )
            
            # For disappeared tracks, be more lenient
            if not is_consistent and track['disappeared'] < 5:
                continue
            
            sim = self.gallery.compare(track_id, detection['emb'])
            if sim > best_score:
                best_score = sim
                best_id = track_id
        
        # Check deleted tracks
        for track_id in self.deleted_tracks.keys():
            # For deleted, only use feature similarity
            sim = self.gallery.compare(track_id, detection['emb'])
            if sim > best_score:
                best_score = sim
                best_id = track_id
        
        return best_id if best_score > REID_THRESHOLD else None
    
    def _create_track(self, detection):
        """Create new track"""
        track_id = self.next_id
        self.next_id += 1
        
        center = bbox_center(detection['box'])
        quality = face_quality_score(detection)
        
        self.active_tracks[track_id] = {
            'box': detection['box'],
            'face': detection['face'],
            'emb': detection['emb'],
            'disappeared': 0,
            'trajectory': deque([tuple(center)], maxlen=60),
            'frames_tracked': 1,
            'total_quality': quality,
            'created_frame': self.frame_count
        }
        
        # Initialize spatial history
        self.position_history[track_id].append(center)
        self.size_history[track_id].append(bbox_size(detection['box']))
        
        self.gallery.add(track_id, detection['emb'], quality, self.frame_count)
    
    def _update_track(self, track_id, detection):
        """Update existing track"""
        track = self.active_tracks[track_id]
        
        track['box'] = detection['box']
        track['face'] = detection['face']
        track['disappeared'] = 0
        track['frames_tracked'] += 1
        
        center = bbox_center(detection['box'])
        track['trajectory'].append(tuple(center))
        
        # Update spatial history
        self.position_history[track_id].append(center)
        if len(self.position_history[track_id]) > 30:
            self.position_history[track_id].popleft()
        
        self.size_history[track_id].append(bbox_size(detection['box']))
        if len(self.size_history[track_id]) > 30:
            self.size_history[track_id].popleft()
        
        quality = face_quality_score(detection)
        track['total_quality'] += quality
        
        self.gallery.add(track_id, detection['emb'], quality, self.frame_count)
    
    def _move_to_deleted(self, track_id):
        """Move track to deleted"""
        self.deleted_tracks[track_id] = {
            'deleted_frame': self.frame_count,
            'data': self.active_tracks[track_id].copy()
        }
        del self.active_tracks[track_id]
    
    def _resurrect_or_update(self, track_id, detection):
        """Resurrect or update track"""
        if track_id in self.deleted_tracks:
            del self.deleted_tracks[track_id]
            
            center = bbox_center(detection['box'])
            quality = face_quality_score(detection)
            
            self.active_tracks[track_id] = {
                'box': detection['box'],
                'face': detection['face'],
                'emb': detection['emb'],
                'disappeared': 0,
                'trajectory': deque([tuple(center)], maxlen=60),
                'frames_tracked': 1,
                'total_quality': quality,
                'created_frame': self.frame_count
            }
            
            # Reset spatial history
            self.position_history[track_id].clear()
            self.size_history[track_id].clear()
            self.position_history[track_id].append(center)
            self.size_history[track_id].append(bbox_size(detection['box']))
            
        elif track_id in self.active_tracks:
            self._update_track(track_id, detection)
        
        quality = face_quality_score(detection)
        self.gallery.add(track_id, detection['emb'], quality, self.frame_count)
    
    def _try_global_match(self, detection):
        """Try to match with any known track"""
        det_center = bbox_center(detection['box'])
        best_id = None
        best_score = 0.0
        
        # Active tracks
        for track_id in self.active_tracks.keys():
            sim = self.gallery.compare(track_id, detection['emb'])
            if sim > best_score:
                best_score = sim
                best_id = track_id
        
        # Deleted tracks
        for track_id in self.deleted_tracks.keys():
            sim = self.gallery.compare(track_id, detection['emb'])
            if sim > best_score:
                best_score = sim
                best_id = track_id
        
        return best_id if best_score > REID_THRESHOLD else None
    
    def draw_tracks(self, frame):
        """Draw all active tracks"""
        for track_id, track in self.active_tracks.items():
            if track['disappeared'] == 0:
                x1, y1, x2, y2 = map(int, track['box'])
                color = self._get_color(track_id)
                
                # Bounding box
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)
                
                # ID label
                label = f"ID:{track_id}"
                (w, h), _ = cv2.getTextSize(label, FONT, 0.8, 2)
                cv2.rectangle(frame, (x1, y1-h-12), (x1+w+10, y1), color, -1)
                cv2.putText(frame, label, (x1+5, y1-5), FONT, 0.8, (255, 255, 255), 2)
                
                # Trajectory
                if len(track['trajectory']) > 1:
                    pts = list(track['trajectory'])
                    for i in range(1, len(pts)):
                        pt1 = tuple(map(int, pts[i-1]))
                        pt2 = tuple(map(int, pts[i]))
                        thickness = int(2 * (i / len(pts)) + 1)
                        cv2.line(frame, pt1, pt2, color, thickness)
        
        return frame

# =============================
# FACE PANEL
# =============================
def create_face_panel(tracker, height):
    """Create side panel"""
    panel = np.ones((height, FACE_PANEL_WIDTH, 3), np.uint8) * 40
    
    cv2.putText(panel, "Tracked", (10, 25), FONT, 0.5, (255, 255, 255), 1)
    cv2.putText(panel, "Faces", (10, 45), FONT, 0.5, (255, 255, 255), 1)
    
    active = [
        (tid, t) for tid, t in tracker.active_tracks.items()
        if t['disappeared'] == 0 and t['face'] is not None
    ]
    active.sort(key=lambda x: x[1]['frames_tracked'], reverse=True)
    
    y = 60
    for track_id, track in active[:MAX_FACES_DISPLAY]:
        if y + FACE_SIZE > height:
            break
        
        face = cv2.resize(track['face'], (FACE_SIZE, FACE_SIZE))
        color = tracker._get_color(track_id)
        
        panel[y:y+FACE_SIZE, 15:15+FACE_SIZE] = face
        cv2.rectangle(panel, (13, y-2), (15+FACE_SIZE+2, y+FACE_SIZE+2), color, 2)
        cv2.putText(panel, f"ID:{track_id}", (15, y-5), FONT, 0.4, color, 1)
        cv2.putText(
            panel, f"#{track['frames_tracked']}f",
            (15, y+FACE_SIZE+15), FONT, 0.35, (200, 200, 200), 1
        )
        
        y += FACE_SIZE + 25
    
    return panel

# =============================
# MAIN
# =============================
print("="*60)
print("Spatial-Aware Face Tracking (Anti ID-Swap)")
print("="*60)
print(f"Video: {YOUTUBE_URL}")
print(f"Time: {START_TIME}s - {END_TIME}s")
print("Initializing...")

yt = YouTube(YOUTUBE_URL)
stream = yt.streams.filter(
    progressive=True, file_extension="mp4"
).order_by('resolution').desc().first()

cap = cv2.VideoCapture(stream.url)
cap.set(cv2.CAP_PROP_POS_MSEC, START_TIME * 1000)

tracker = SpatialAwareFaceTracker()
frame_count = 0

print("Starting tracking with spatial consistency...")
print("-"*60)

while cap.isOpened():
    clear_output(wait=True)
    ret, frame = cap.read()
    if not ret:
        break
    
    current_time = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000
    if current_time >= END_TIME:
        break
    
    frame_count += 1
    
    # Detect faces
    faces_data = app.get(frame)
    
    # Prepare detections
    detections = []
    for face in faces_data:
        if face.det_score < MIN_CONFIDENCE:
            continue
        
        box = face.bbox.astype(int)
        
        if bbox_size(box) < MIN_FACE_SIZE * MIN_FACE_SIZE:
            continue
        
        face_crop = extract_face(frame, box)
        if face_crop is None:
            continue
        
        embedding = face.embedding / (np.linalg.norm(face.embedding) + 1e-8)
        
        detections.append({
            'box': box,
            'face': face_crop,
            'emb': embedding,
            'score': face.det_score
        })
    
    # Update tracker
    tracker.update(detections, frame)
    
    # Draw
    frame = tracker.draw_tracks(frame)
    
    # Info
    info_text = f"Frame: {frame_count} | Time: {current_time:.1f}s"
    cv2.putText(frame, info_text, (10, 30), FONT, 0.7, (0, 255, 0), 2)
    
    if tracker.scene_changed:
        cv2.putText(frame, "SCENE CHANGE", (10, 60), FONT, 0.7, (0, 0, 255), 2)
    
    # Resize
    h, w = frame.shape[:2]
    new_h = int(TARGET_WIDTH * h / w)
    frame = cv2.resize(frame, (TARGET_WIDTH, new_h))
    
    # Panel
    panel = create_face_panel(tracker, new_h)
    combined = np.hstack([frame, panel])
    
    # Display
    _, buffer = cv2.imencode('.jpg', combined)
    display(Image(data=buffer.tobytes()))
    
    time.sleep(0.02)

cap.release()

# Statistics
print("\n" + "="*60)
print("TRACKING STATISTICS")
print("="*60)
print(f"Processed frames: {frame_count}")
print(f"Total unique faces: {tracker.next_id - 1}")
print(f"Active tracks: {len(tracker.active_tracks)}")
print(f"Deleted tracks in memory: {len(tracker.deleted_tracks)}")
print("-"*60)

if len(tracker.active_tracks) > 0:
    print("Active track details:")
    for track_id, track in sorted(
        tracker.active_tracks.items(),
        key=lambda x: x[1]['frames_tracked'],
        reverse=True
    ):
        print(f"  ID {track_id}: {track['frames_tracked']} frames")

print("="*60)

KeyboardInterrupt: 